# Notebook 1 — Creem Webhook: Purchase → License Provisioning

**Purpose:** Test the webhook flow where Creem.io fires a `checkout.completed` event after a customer pays, and Railway provisions the license key into Supabase.

---

## Architecture

```
Customer pays on Creem.io
        │
        ▼
Creem fires POST /v1/webhooks/creem  (with HMAC-SHA256 signature)
        │
        ▼
Railway Backend (license_api_production.py)
  1. Verifies x-creem-signature header (HMAC-SHA256)
  2. Parses event type + license_key from payload
  3. Upserts license row into Supabase  (is_active=true)
        │
        ▼
License is now in Supabase — ready for validation by the desktop app
```

## Lifecycle Events Tested

| Step | Creem Event | Expected Supabase Effect |
|------|-------------|-------------------------|
| 0 | — | Setup: load env, connect clients |
| 1 | — | Railway health check |
| 2 | `checkout.completed` | New license row upserted, `is_active=true` |
| 3 | — | Read-back: confirm the row exists in Supabase |
| 4 | `subscription.cancelled` | `is_active` flipped to `false` |
| 5 | `subscription.renewed` | `is_active` flipped back to `true`, expiry extended |
| 6 | `charge.refunded` | `is_active` set to `false` |
| 7 | — | Cleanup: remove all test rows |

> **Mode:** Creem **Test / Sandbox** mode (`CREEM_API_MODE=sandbox` → `https://test-api.creem.io`)
>
> **Prerequisite:** `pip install httpx python-dotenv supabase`

---

## Environment Variables Required

These must exist in your `.env` file (`Orchestrator/.env`):

| Variable | Where to find it | Purpose |
|----------|------------------|---------|
| `SUPABASE_URL` | Supabase → Project Settings → API → URL | Database connection |
| `SUPABASE_SERVICE_ROLE_KEY` | Supabase → Project Settings → API → service_role key | Full DB access |
| `CREEM_TEST_API_KEY` | Creem Dashboard → Developers → API Keys (test mode) | Server-side API calls |
| `CREEM_TEST_API_URL` | Already in .env: `https://test-api.creem.io` | Sandbox API base URL |
| `CREEM_PRODUCT_ID` | Creem Dashboard → Products → copy ID | Product reference |

### Not yet in .env (action required):

| Variable | Where to find it | Purpose |
|----------|------------------|---------|
| `CREEM_WEBHOOK_SECRET` | Creem Dashboard → Developers → Webhooks → signing secret (`whsec_...`) | HMAC signature verification |

> ⚠️ Until `CREEM_WEBHOOK_SECRET` is added, webhook signature verification is bypassed.

### Also required in Railway environment variables:

| Variable | Value |
|----------|-------|
| `CREEM_API_KEY` | Same value as `CREEM_TEST_API_KEY` (for test mode) |
| `CREEM_WEBHOOK_SECRET` | The `whsec_...` value from Creem |
| `CREEM_API_MODE` | `sandbox` (for testing) |

## Step 0 — Setup

Load environment variables, initialise the Supabase client, and define test constants.
All test data uses a unique prefix (`CRMTEST-`) so it never collides with real licenses.

In [1]:
import os, json, time, hmac, hashlib
import httpx
from dotenv import load_dotenv
from supabase import create_client

# ── Load your .env ─────────────────────────────────────────────────────────
# Your .env contains: SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY,
#   CREEM_TEST_API_KEY, CREEM_TEST_API_URL, CREEM_PRODUCT_ID
load_dotenv(r"C:/Users/sossi/Desktop/Business/Orchestrator Hedge Edge/Orchestrator/.env")

# ── Railway backend URL ────────────────────────────────────────────────────
RAILWAY_URL = "https://hedgeedge-railway-backend-production.up.railway.app"

# ── Supabase client (direct DB access for verification) ───────────────────
SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_KEY = os.environ["SUPABASE_SERVICE_ROLE_KEY"]
db = create_client(SUPABASE_URL, SUPABASE_KEY)

# ── Creem config ──────────────────────────────────────────────────────────
# Your .env has separate keys for live vs test mode.
# We use TEST keys since we're testing in Creem sandbox.
CREEM_TEST_API_KEY   = os.environ.get("CREEM_TEST_API_KEY", "")
CREEM_TEST_API_URL   = os.environ.get("CREEM_TEST_API_URL", "https://test-api.creem.io")
CREEM_WEBHOOK_SECRET = os.environ.get("CREEM_WEBHOOK_SECRET", "")
CREEM_PRODUCT_ID     = os.environ.get("CREEM_PRODUCT_ID", "")
# ⚠️ CREEM_WEBHOOK_SECRET is not in your .env yet!
# Get it from: Creem Dashboard → Developers → Webhooks → signing secret (whsec_...)
# Once you have it, add to .env:  CREEM_WEBHOOK_SECRET=whsec_your_secret_here

# ── Test constants (created + cleaned up within this notebook) ────────────
TEST_LICENSE_KEY = "CRMTEST-PROV-0001-00001-00001"
TEST_EMAIL       = "creem-webhook-test@hedgeedge.com"

print("Railway URL      :", RAILWAY_URL)
print("Supabase URL     :", SUPABASE_URL[:50])
print("Creem test API   :", CREEM_TEST_API_URL)
print("Creem test key   :", "SET" if CREEM_TEST_API_KEY else "NOT SET")
print("Creem product ID :", CREEM_PRODUCT_ID[:12] + "..." if CREEM_PRODUCT_ID else "NOT SET")
print("Webhook secret   :", "SET" if CREEM_WEBHOOK_SECRET else "NOT SET — signature bypassed")
print("Test license key :", TEST_LICENSE_KEY)
print()
print("OK ✅  Setup complete")

Railway URL      : https://hedgeedge-railway-backend-production.up.railway.app
Supabase URL     : https://qgaikasjcfyfjcxezfqk.supabase.co
Creem test API   : https://test-api.creem.io
Creem test key   : SET
Creem product ID : prod_1us2Psu...
Webhook secret   : NOT SET — signature bypassed
Test license key : CRMTEST-PROV-0001-00001-00001

OK ✅  Setup complete


## Step 1 — Railway Health Check

Confirm the Railway backend is online before sending webhook payloads.
The `/health` endpoint returns `{"status": "healthy"}` and includes `serverTime` for clock-drift detection.

In [2]:
r = httpx.get(f"{RAILWAY_URL}/health", timeout=10)
data = r.json()

print(f"HTTP {r.status_code}")
print(f"Version    : {data.get('version')}")
print(f"Status     : {data.get('status')}")
print(f"Server Time: {data.get('timestamp')}")

assert r.status_code == 200, f"Expected 200, got {r.status_code}"
assert data.get("status") == "healthy", f"Expected healthy, got {data.get('status')}"
print("\nOK ✅  Railway is healthy")

HTTP 200
Version    : 2.1.0
Status     : healthy
Server Time: 2026-03-12T17:39:38.717342+00:00

OK ✅  Railway is healthy


## Step 2 — Simulate `checkout.completed` (New Purchase)

This simulates what Creem.io sends when a customer completes a payment:

1. **Creem fires** a POST to `/v1/webhooks/creem` with a JSON body containing `type: checkout.completed`
2. **Railway verifies** the `x-creem-signature` HMAC header
3. **Railway upserts** the license into Supabase with `is_active=true`

### Signature Generation

We compute the HMAC-SHA256 signature locally using `CREEM_WEBHOOK_SECRET`.
This is the same algorithm Railway uses to verify incoming webhooks.

> If `CREEM_WEBHOOK_SECRET` is not set in Railway, signature verification is bypassed
> and we send `x-creem-signature: test` as a placeholder.

In [3]:
# ── Clean up any leftover test row so the test is idempotent ──────────────
db.table("licenses").delete().eq("license_key", TEST_LICENSE_KEY).execute()

# ── Build the webhook payload (matches Creem's actual format) ─────────────
webhook_body = json.dumps({
    "type": "checkout.completed",
    "data": {
        "license_key": TEST_LICENSE_KEY,
        "customer": {"email": TEST_EMAIL},
        "product":  {"name": "Hedge Edge Pro"},
        "license":  {"expires_at": "2027-12-31T00:00:00Z"}
    }
}).encode()

# ── Compute HMAC-SHA256 signature (same algo as backend verification) ─────
if CREEM_WEBHOOK_SECRET:
    computed_sig = hmac.new(
        CREEM_WEBHOOK_SECRET.encode(),
        webhook_body,
        hashlib.sha256
    ).hexdigest()
    print(f"Computed HMAC signature: {computed_sig[:24]}...")
else:
    computed_sig = "test"  # Bypass when secret is not configured
    print("⚠️  No CREEM_WEBHOOK_SECRET — using placeholder signature")

headers = {
    "Content-Type": "application/json",
    "x-creem-signature": computed_sig,
}

# ── Fire the webhook at Railway ──────────────────────────────────────────
r = httpx.post(
    f"{RAILWAY_URL}/v1/webhooks/creem",
    content=webhook_body,
    headers=headers,
    timeout=15
)
data = r.json()

print(f"\nHTTP {r.status_code}")
print(json.dumps(data, indent=2))

assert r.status_code == 200, f"Expected 200, got {r.status_code}"
assert data.get("processed") == True, f"Expected processed=true, got {data}"
assert data.get("action") == "provisioned", f"Expected action=provisioned, got {data.get('action')}"
print("\nOK ✅  Railway processed checkout.completed → license provisioned")

⚠️  No CREEM_WEBHOOK_SECRET — using placeholder signature

HTTP 200
{
  "received": true,
  "processed": true,
  "action": "provisioned",
  "plan": "professional"
}

OK ✅  Railway processed checkout.completed → license provisioned


## Step 3 — Verify License Exists in Supabase

Read back the license row directly from Supabase to confirm:
- The row was created
- `is_active` is `true`
- `email`, `plan`, and `expires_at` are populated correctly

This proves the webhook actually wrote to the database (not just returned 200 OK).

In [4]:
row = db.table("licenses") \
    .select("license_key, email, plan, is_active, expires_at, created_at") \
    .eq("license_key", TEST_LICENSE_KEY) \
    .single() \
    .execute()

lic = row.data
print("License record in Supabase:")
print(f"  Key        : {lic['license_key']}")
print(f"  Email      : {lic['email']}")
print(f"  Plan       : {lic['plan']}")
print(f"  Active     : {lic['is_active']}")
print(f"  Expires    : {lic['expires_at']}")
print(f"  Created    : {lic['created_at']}")

assert lic["is_active"] == True, f"Expected is_active=True, got {lic['is_active']}"
assert lic["email"] == TEST_EMAIL, f"Expected email={TEST_EMAIL}, got {lic['email']}"
print(f"\nOK ✅  License confirmed in Supabase — plan={lic['plan']}, expires={lic['expires_at']}")

License record in Supabase:
  Key        : CRMTEST-PROV-0001-00001-00001
  Email      : creem-webhook-test@hedgeedge.com
  Plan       : professional
  Active     : True
  Expires    : 2027-12-31T00:00:00+00:00
  Created    : 2026-03-12T17:39:54.451195+00:00

OK ✅  License confirmed in Supabase — plan=professional, expires=2027-12-31T00:00:00+00:00


## Step 3b — Schema Fix (Idempotent)

The backend's cancellation/refund handler writes `deactivated_at` to the `licenses` table.
This column was defined in the migration file (`20260201000000_license_system.sql`) but was missing from the live database.
This cell adds it using `IF NOT EXISTS` — safe to re-run on every test.

In [ ]:
# ── Schema Fix: Ensure deactivated_at column exists on licenses table ─────
# The backend webhook handler sets deactivated_at on cancellation/refund events.
# This column was defined in the migration but not applied to the live DB.
# This cell is idempotent (IF NOT EXISTS) — safe to re-run.

SUPABASE_ACCESS_TOKEN = os.environ.get("SUPABASE_ACCESS_TOKEN", "")
PROJECT_REF = SUPABASE_URL.replace("https://", "").split(".")[0]

if not SUPABASE_ACCESS_TOKEN:
    print("SUPABASE_ACCESS_TOKEN not set — add column manually via SQL Editor:")
    print("  ALTER TABLE public.licenses ADD COLUMN IF NOT EXISTS deactivated_at TIMESTAMPTZ;")
else:
    r = httpx.post(
        f"https://api.supabase.com/v1/projects/{PROJECT_REF}/database/query",
        headers={
            "Authorization": f"Bearer {SUPABASE_ACCESS_TOKEN}",
            "Content-Type": "application/json",
        },
        json={"query": "ALTER TABLE public.licenses ADD COLUMN IF NOT EXISTS deactivated_at TIMESTAMPTZ;"},
        timeout=15,
    )
    if r.status_code in (200, 201):
        print("OK ✅  deactivated_at column present on licenses table")
    else:
        print(f"HTTP {r.status_code}: {r.text}")

OK - deactivated_at column added to licenses table
Verified: deactivated_at = None


## Step 4 — Simulate `subscription.cancelled` (Deactivation)

When a customer cancels their subscription in Creem:
1. Creem fires `subscription.cancelled` webhook
2. Railway sets `is_active = false` and records `deactivated_at`
3. Subsequent license validations from the desktop app will be rejected

This is the critical path — cancellation must propagate within seconds.

In [8]:
cancel_body = json.dumps({
    "type": "subscription.cancelled",
    "data": {"license_key": TEST_LICENSE_KEY}
}).encode()

# Compute signature for this payload
if CREEM_WEBHOOK_SECRET:
    sig = hmac.new(CREEM_WEBHOOK_SECRET.encode(), cancel_body, hashlib.sha256).hexdigest()
else:
    sig = "test"

r = httpx.post(
    f"{RAILWAY_URL}/v1/webhooks/creem",
    content=cancel_body,
    headers={"Content-Type": "application/json", "x-creem-signature": sig},
    timeout=15
)
data = r.json()

print(f"HTTP {r.status_code}")
print(json.dumps(data, indent=2))

assert r.status_code == 200
assert data.get("action") == "deactivated"

# ── Confirm in Supabase ────────────────────────────────────────────────────
row = db.table("licenses").select("is_active").eq("license_key", TEST_LICENSE_KEY).single().execute()
assert row.data["is_active"] == False, f"Expected is_active=False, still True!"
print("\nOK ✅  is_active=False confirmed in Supabase")

HTTP 200
{
  "received": true,
  "processed": true,
  "action": "deactivated",
  "affected": 1
}

OK ✅  is_active=False confirmed in Supabase


## Step 5 — Simulate `subscription.renewed` (Reactivation)

When the customer resubscribes or their payment succeeds again:
1. Creem fires `subscription.renewed` webhook with an updated `expires_at`
2. Railway sets `is_active = true` and updates the expiry date
3. The license is now valid again for desktop app validation

In [9]:
renew_body = json.dumps({
    "type": "subscription.renewed",
    "data": {
        "license_key": TEST_LICENSE_KEY,
        "expires_at": "2028-12-31T00:00:00Z"
    }
}).encode()

if CREEM_WEBHOOK_SECRET:
    sig = hmac.new(CREEM_WEBHOOK_SECRET.encode(), renew_body, hashlib.sha256).hexdigest()
else:
    sig = "test"

r = httpx.post(
    f"{RAILWAY_URL}/v1/webhooks/creem",
    content=renew_body,
    headers={"Content-Type": "application/json", "x-creem-signature": sig},
    timeout=15
)
data = r.json()

print(f"HTTP {r.status_code}")
print(json.dumps(data, indent=2))

assert r.status_code == 200
assert data.get("action") == "reactivated"

# ── Confirm in Supabase ────────────────────────────────────────────────────
row = db.table("licenses").select("is_active, expires_at").eq("license_key", TEST_LICENSE_KEY).single().execute()
assert row.data["is_active"] == True
print(f"\nOK ✅  is_active=True, new expires_at={row.data['expires_at']}")

HTTP 200
{
  "received": true,
  "processed": true,
  "action": "reactivated",
  "affected": 1
}

OK ✅  is_active=True, new expires_at=2028-12-31T00:00:00+00:00


## Step 6 — Simulate `charge.refunded` (Refund → Deactivation)

When a customer requests a refund:
1. Creem fires `charge.refunded` webhook
2. Railway sets `is_active = false`
3. Access is immediately revoked

This is identical to cancellation from the database perspective.

In [10]:
refund_body = json.dumps({
    "type": "charge.refunded",
    "data": {"license_key": TEST_LICENSE_KEY}
}).encode()

if CREEM_WEBHOOK_SECRET:
    sig = hmac.new(CREEM_WEBHOOK_SECRET.encode(), refund_body, hashlib.sha256).hexdigest()
else:
    sig = "test"

r = httpx.post(
    f"{RAILWAY_URL}/v1/webhooks/creem",
    content=refund_body,
    headers={"Content-Type": "application/json", "x-creem-signature": sig},
    timeout=15
)
data = r.json()

print(f"HTTP {r.status_code}")
print(json.dumps(data, indent=2))

assert r.status_code == 200
assert data.get("action") == "deactivated"

# ── Confirm in Supabase ────────────────────────────────────────────────────
row = db.table("licenses").select("is_active").eq("license_key", TEST_LICENSE_KEY).single().execute()
assert row.data["is_active"] == False
print("\nOK ✅  Refund deactivated license — is_active=False confirmed")

HTTP 200
{
  "received": true,
  "processed": true,
  "action": "deactivated",
  "affected": 1
}

OK ✅  Refund deactivated license — is_active=False confirmed


## Step 7 — Cleanup

Remove all test rows from Supabase so the database is clean.
Cascading deletes: sessions → devices → license.

In [11]:
# ── Cascade: delete sessions and devices first ────────────────────────────
lic_row = db.table("licenses").select("id").eq("license_key", TEST_LICENSE_KEY).execute()

if lic_row.data:
    lic_id = lic_row.data[0]["id"]
    sessions = db.table("license_sessions").delete().eq("license_id", lic_id).execute()
    devices  = db.table("license_devices").delete().eq("license_id", lic_id).execute()
    print(f"Deleted {len(sessions.data)} session(s)")
    print(f"Deleted {len(devices.data)} device(s)")

# ── Delete the license row ────────────────────────────────────────────────
lic = db.table("licenses").delete().eq("license_key", TEST_LICENSE_KEY).execute()
print(f"Deleted {len(lic.data)} license row(s)")
print("\nOK ✅  Cleanup complete — Supabase is back to its original state")

Deleted 0 session(s)
Deleted 0 device(s)
Deleted 1 license row(s)

OK ✅  Cleanup complete — Supabase is back to its original state


## Summary

| # | Test | Event | Pass Condition |
|---|------|-------|----------------|
| 0 | Setup | — | Config loaded, Supabase + Creem clients ready |
| 1 | Health check | — | Railway returns `status=healthy` |
| 2 | Purchase | `checkout.completed` | `action=provisioned`, row created in Supabase |
| 3 | Read-back | — | License row has `is_active=true`, correct email/plan |
| 4 | Cancellation | `subscription.cancelled` | `action=deactivated`, `is_active=false` in DB |
| 5 | Renewal | `subscription.renewed` | `action=reactivated`, `is_active=true`, expiry extended |
| 6 | Refund | `charge.refunded` | `action=deactivated`, `is_active=false` in DB |
| 7 | Cleanup | — | All test rows removed |

---

### Production Checklist

| Task | Status |
|------|--------|
| Set `CREEM_WEBHOOK_SECRET` in Railway env vars (from Creem → Developers → Webhooks) | ⬜ |
| Set `CREEM_API_KEY` in Railway env vars (from Creem → Developers → API Keys) | ⬜ |
| Set `CREEM_API_MODE=production` in Railway env vars (currently `sandbox`) | ⬜ |
| Register webhook URL in Creem: `https://hedgeedge-railway-backend-production.up.railway.app/v1/webhooks/creem` | ⬜ |
| Verify webhook fires in Creem test mode before going live | ⬜ |